# 11c — Outcomes derivados: produtividade canavieira e intensidade de emissão por área agrícola

**Versão 1** (5 maio 2026)

**Escopo da Fase B4 inicial (§11.2 v2.3.6+):**
- **Outcome 1 — `log_produtividade_cana`** = $\log(\text{qtd\_produzida\_t\_pam} / \text{area\_colhida\_ha\_pam})$. Mede produtividade canavieira observada (T/ha de cana). **Teste crítico para a hipótese (c) da §10.3 v2.3.6:** se municípios certificados expandiram produção sem expandir área (LUC nulo confirmado), então produtividade DEVE estar subindo. Sinal positivo robusto = hipótese (c) ganha. Sinal nulo = hipóteses (a) ou (d) ganham (gap declaração-realidade).
- **Outcome 2 — `log_emissao_solos_por_ha_agricola`** = $\log(\text{solos\_manejados} / \text{pam\_area\_colhida\_total})$. Mede intensidade observada de emissão por área agrícola. **Complemento ao achado central:** se o ATT positivo em emissão absoluta vier acompanhado de ATT positivo nesta intensidade normalizada, então o efeito não é só de escala — há piora real de intensidade observada (compatível com hipótese (a) ou (d)).

**Decisão SICAR — adiada para Fase B4-SICAR-dedicada:** a variável `cobertura_car_ativo` no painel apresenta não-monotonicidade ano-a-ano em ≥84% da amostra investigada (168/200 munis com pelo menos uma queda > 0,001), contradizendo a premissa física de que a cobertura cadastral SICAR só pode aumentar. A causa provável é inconsistência na construção da agregação anual a partir dos snapshots mensais SICAR. Sem investigação ETL própria, a estimação CS-DR sobre `cobertura_car_ativo` produziria estimativas não confiáveis. Esta sub-etapa fica explicitamente declarada como pendência de Fase B4 dedicada, com diagnóstico já documentado em v2.3.7 §11.2.

**Estimadores e specs:** mesmo schema dos *notebooks* 11a v4 e 11b v1 — CS-DR (principal, 4 specs LEAN/FULL/FULL−2/RICH) + Sun-Abraham canônico (referência staggered heterogeneity-robust) + TWFE clássico (referência ingênua). Bootstrap multiplicador *n=199* para CS-DR; clustered CR1 por município para Sun-Abraham/TWFE.

**Total a estimar:** 4 specs × 2 outcomes × 1 estimador (CS-DR) + 2 outcomes × 1 estimador (Sun-Abraham canônico) + 2 outcomes × 1 estimador (TWFE) = **12 ATTs**.

**Tempo esperado:** ~3-5 min com n_boot=199.

**Inputs:**
- `data/interim/panel_canavieiro_main.csv`
- `data/raw/psm_baseline/base_psm_integrada_raw.csv`

**Outputs em `data/interim/`:**
- `att_derived_outcomes.csv` — 12 ATTs
- `att_derived_eventstudy.csv` — event-study CS-DR sob FULL para os 2 outcomes

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path
BASE_DIR = Path('/content/drive/MyDrive/Renovabio - EcoEco')
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

!pip install -q differences pyfixest linearmodels

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from differences import ATTgt
import pyfixest as pf
from linearmodels.panel import PanelOLS

from pipeline.config import interim, out_pre

N_BOOT = 199
RANDOM_STATE = 42

print(f'✓ setup OK (bootstrap n={N_BOOT}, seed={RANDOM_STATE})')

## Bloco 1 — Carregar painel e construir outcomes derivados

In [ ]:
panel = pd.read_csv(interim('panel_canavieiro_main.csv'), dtype={'geocode': str})

# Dropar covs colidentes (mesma estratégia 11a/11b)
COVS_COLIDENTES = ['gini', 'densidade_pop', 'log_pop', 'idhm_renda',
                    'ivs_capital_humano', 'ivs_renda_trabalho']
panel = panel.drop(columns=[c for c in COVS_COLIDENTES if c in panel.columns])
print(f'panel após dropar colidentes: {panel.shape}')

# === Outcome 1: log_produtividade_cana ===
panel['produtividade_cana_t_ha'] = np.where(
    (panel['area_colhida_ha_pam'].notna()) & (panel['area_colhida_ha_pam'] > 0),
    panel['qtd_produzida_t_pam'] / panel['area_colhida_ha_pam'],
    np.nan
)
panel['log_produtividade_cana'] = np.log(panel['produtividade_cana_t_ha'].replace(0, np.nan))

# === Outcome 2: log_emissao_solos_por_ha_agricola ===
# log_solos_manejados está em log puro (não log1p), então exp() retorna emissão absoluta
panel['emissao_solos_manejados'] = np.exp(panel['log_solos_manejados'])
panel['emissao_solos_por_ha_agricola'] = np.where(
    (panel['pam_area_colhida_total'].notna()) & (panel['pam_area_colhida_total'] > 0),
    panel['emissao_solos_manejados'] / panel['pam_area_colhida_total'],
    np.nan
)
panel['log_emissao_solos_por_ha_agricola'] = np.log(
    panel['emissao_solos_por_ha_agricola'].replace(0, np.nan)
)

# Diagnóstico
print('\n=== Diagnóstico dos 2 outcomes derivados ===')
for o in ['log_produtividade_cana', 'log_emissao_solos_por_ha_agricola']:
    nn = panel[o].notna().sum()
    sub = panel[o].dropna()
    print(f'  {o:40s}: notna={nn}/{len(panel)} ({100*nn/len(panel):.1f}%)')
    print(f'    range log: [{sub.min():.3f}, {sub.max():.3f}], mediana={sub.median():.3f}')
    sub_raw = panel[o.replace('log_', '').replace('cana', 'cana_t_ha').replace('agricola', 'agricola')].dropna() if o == 'log_produtividade_cana' else panel['emissao_solos_por_ha_agricola'].dropna()
    print(f'    range nativo: [{sub_raw.min():.3f}, {sub_raw.max():.3f}], mediana={sub_raw.median():.3f}')

print(f'\n✓ Outcomes derivados construídos')

## Bloco 2 — Reconstruir covariáveis (mesma função 10/11a/11b)

In [ ]:
psm_raw = pd.read_csv(
    BASE_DIR / 'data/raw/psm_baseline/base_psm_integrada_raw.csv',
    low_memory=False,
)
psm_raw['geocode'] = psm_raw['0_cd_ibge'].astype(str).str.zfill(7)

BIOMA_FIXES = {'Amaz\ufffd\ufffdnia': 'Amazônia', 'Mata Atl\ufffd\ufffdntica': 'Mata Atlântica'}
if '14_bioma' in psm_raw.columns:
    psm_raw['14_bioma'] = psm_raw['14_bioma'].replace(BIOMA_FIXES)

muni_id = (panel.groupby('geocode', as_index=False)
           .agg(municipio=('municipio','first'), uf=('uf','first'),
                is_treated_ever=('is_treated_ever','first'), g_m=('g_m','first'),
                bioma=('bioma','first')))
muni_id['treated'] = muni_id['is_treated_ever'].astype(int)
df_cs = muni_id.merge(psm_raw, on='geocode', how='inner')
print(f'df_cs: {df_cs.shape}')

In [ ]:
# Helpers e build_covariates_raw (idêntico a 11a/11b)
def safe_log1p(s, idx):
    s = pd.to_numeric(s, errors='coerce') if s is not None else pd.Series(np.nan, index=idx)
    return np.log1p(s.clip(lower=0))
def safe_div(num, den, idx):
    num = pd.to_numeric(num, errors='coerce') if num is not None else pd.Series(np.nan, index=idx)
    den = pd.to_numeric(den, errors='coerce') if den is not None else pd.Series(np.nan, index=idx)
    return np.where((den.notna()) & (den > 0), num / den, np.nan)
def asn(s, idx):
    return pd.to_numeric(s, errors='coerce') if s is not None else pd.Series(np.nan, index=idx)

def build_covariates_raw(df):
    d = df.copy(); idx = d.index
    d['log_pib_total']=safe_log1p(d.get('1_pib_total'),idx)
    d['log_pib_pc']=safe_log1p(d.get('1_pib_percap'),idx)
    d['log_pop']=safe_log1p(d.get('2_pop_2017_ibge'),idx)
    d['log_area_total']=safe_log1p(d.get('14_area_total'),idx)
    d['densidade_pop']=safe_div(d.get('2_pop_2017_ibge'),d.get('14_area_total'),idx)
    d['share_vadc_agro']=safe_div(d.get('1_vadc_agro'),d.get('1_vadc_bruto'),idx)
    d['share_vadc_ind']=safe_div(d.get('1_vadc_ind'),d.get('1_vadc_bruto'),idx)
    d['share_vadc_serv']=safe_div(d.get('1_vadc_serv'),d.get('1_vadc_bruto'),idx)
    d['share_vadc_adm']=safe_div(d.get('1_vadc_adm'),d.get('1_vadc_bruto'),idx)
    d['share_cana_baseline']=asn(d.get('3_mb_sharegrp_pre_cana'),idx)
    d['mb_share_soja']=asn(d.get('3_mb_sharegrp_pre_soja'),idx)
    d['mb_share_pastagem']=asn(d.get('3_mb_sharegrp_pre_pastagem'),idx)
    d['mb_share_vegetacao_nativa']=asn(d.get('3_mb_sharegrp_pre_vegetacao_nativa'),idx)
    d['mb_share_urbano']=asn(d.get('3_mb_sharegrp_pre_urbano_infra'),idx)
    d['log_area_cana']=safe_log1p(d.get('4_area_colhida_ha_cana'),idx)
    d['log_area_soja']=safe_log1p(d.get('4_area_colhida_ha_soja'),idx)
    d['log_area_agri_total']=safe_log1p(d.get('4_area_colhida_ha'),idx)
    d['share_area_cana_agri']=safe_div(d.get('4_area_colhida_ha_cana'),d.get('4_area_colhida_ha'),idx)
    d['share_est_af']=safe_div(d.get('5_num_est_af'),d.get('5_num_est_total'),idx)
    d['share_est_mp']=safe_div(d.get('5_num_est_mp'),d.get('5_num_est_total'),idx)
    d['share_area_af']=safe_div(d.get('6_area_lav_af'),d.get('6_area_lav_total'),idx)
    d['share_area_mp']=safe_div(d.get('6_area_lav_mp'),d.get('6_area_lav_total'),idx)
    d['trator_per_est']=safe_div(d.get('11_num_trator_total'),d.get('5_num_est_total'),idx)
    d['share_est_irrig']=safe_div(d.get('12_num_est_irrig_total'),d.get('5_num_est_total'),idx)
    d['share_area_irrig']=safe_div(d.get('12_area_irrig_total'),d.get('6_area_lav_total'),idx)
    d['share_est_fin_total']=safe_div(d.get('13_num_est_fin_total'),d.get('5_num_est_total'),idx)
    d['share_est_at']=safe_div(d.get('10_num_est_receb_at'),d.get('5_num_est_total'),idx)
    d['natveg_share_area']=safe_div(d.get('14_vegetacao_natural'),d.get('14_area_total'),idx)
    d['desmat_share_area']=safe_div(d.get('14_desmatado'),d.get('14_area_total'),idx)
    d['idhm_renda']=asn(d.get('17_idhm_renda'),idx); d['idhm_educ']=asn(d.get('17_idhm_educ'),idx)
    d['ivs_infra']=asn(d.get('17_ivs_infraestrutura_urbana'),idx); d['gini']=asn(d.get('17_i_gini'),idx)
    d['idhm_long']=asn(d.get('17_idhm_long'),idx); d['ivs_capital_humano']=asn(d.get('17_ivs_capital_humano'),idx)
    d['ivs_renda_trabalho']=asn(d.get('17_ivs_renda_e_trabalho'),idx)
    d['share_est_at_coop']=safe_div(d.get('10_num_est_receb_at_coop'),d.get('5_num_est_total'),idx)
    d['share_est_at_gov']=safe_div(d.get('10_num_est_receb_at_gov'),d.get('5_num_est_total'),idx)
    d['share_fin_invest']=safe_div(d.get('13_num_est_fin_invest'),d.get('5_num_est_total'),idx)
    d['share_fin_cust']=safe_div(d.get('13_num_est_fin_cust'),d.get('5_num_est_total'),idx)
    d['share_est_trator']=safe_div(d.get('11_num_est_trator_total'),d.get('5_num_est_total'),idx)
    d['share_est_irrig_pivo']=safe_div(d.get('12_num_est_irrig_pivo'),d.get('5_num_est_total'),idx)
    d['pct_est_energia']=asn(d.get('7_est_com_energia%'),idx)
    d['log_area_milho']=safe_log1p(d.get('4_area_colhida_ha_milho'),idx)
    d['log_area_alg']=safe_log1p(d.get('4_area_colhida_ha_alg'),idx)
    d['log_area_cafarab']=safe_log1p(d.get('4_area_colhida_ha_cafarab'),idx)
    d['mb_share_agua']=asn(d.get('3_mb_sharegrp_pre_agua'),idx)
    d['mb_share_outros']=asn(d.get('3_mb_sharegrp_pre_outros'),idx)
    d['mb_share_agri_total']=asn(d.get('3_mb_sharegrp_pre_agricultura_total'),idx)
    d['share_num_est_mp']=safe_div(d.get('5_num_est_mp'),d.get('5_num_est_total'),idx)
    d['share_est_pec']=safe_div(d.get('5_num_est_pec_total'),d.get('5_num_est_total'),idx)
    d['share_est_lavperm']=safe_div(d.get('5_num_est_lavperm_total'),d.get('5_num_est_total'),idx)
    d['share_est_lavtemp']=safe_div(d.get('5_num_est_lavtemp_total'),d.get('5_num_est_total'),idx)
    d['share_area_lavperm']=safe_div(d.get('6_area_lavperm_total'),d.get('6_area_lav_total'),idx)
    d['share_area_lavtemp']=safe_div(d.get('6_area_lavtemp_total'),d.get('6_area_lav_total'),idx)
    d['share_area_pec']=safe_div(d.get('6_area_pec_total'),d.get('6_area_lav_total'),idx)
    d['share_fin_comer']=safe_div(d.get('13_num_est_fin_comer'),d.get('5_num_est_total'),idx)
    d['share_est_at_propr']=safe_div(d.get('10_num_est_receb_at_propr'),d.get('5_num_est_total'),idx)
    d['share_est_at_gov_out']=safe_div(d.get('10_num_est_receb_at_gov_out'),d.get('5_num_est_total'),idx)
    share_cols = [c for c in d.columns if ('share' in c) or c.startswith('mb_share_')]
    for c in share_cols:
        s = pd.to_numeric(d[c], errors='coerce')
        if not s.dropna().empty and (s.dropna().between(-0.05,1.05).mean() > 0.8):
            d[c] = s.clip(0,1)
    return d

df_cs = build_covariates_raw(df_cs)
print(f'✓ Covariáveis reconstruídas')

In [ ]:
# Specs PSM idênticas a 11a v4 / 11b v1 — RICH-52 v2.3.5
COVS_LEAN = ['log_pib_total','log_pib_pc','log_pop','densidade_pop',
             'share_vadc_agro','share_vadc_ind','share_vadc_serv',
             'share_cana_baseline','mb_share_soja','mb_share_pastagem','mb_share_vegetacao_nativa',
             'log_area_cana','log_area_soja','share_area_cana_agri',
             'share_est_af','share_area_af','trator_per_est','share_est_irrig','share_est_fin_total',
             'ivs_infra','gini']
COVS_FULL = COVS_LEAN + ['share_vadc_adm','idhm_educ','idhm_renda','idhm_long',
                          'ivs_capital_humano','ivs_renda_trabalho',
                          'share_est_at','share_est_at_coop','share_est_at_gov',
                          'share_fin_invest','share_fin_cust',
                          'share_est_trator','share_est_irrig_pivo','pct_est_energia']
COVS_FULL2 = [c for c in COVS_FULL if c not in ('share_vadc_agro','share_vadc_ind')]
COVS_RICH = COVS_FULL + [c for c in [
    'log_area_milho','log_area_alg','log_area_cafarab','mb_share_urbano','mb_share_agua',
    'mb_share_outros','mb_share_agri_total','share_num_est_mp','share_est_pec',
    'share_est_lavperm','share_est_lavtemp','share_area_lavperm','share_area_lavtemp',
    'share_area_pec','share_fin_comer','share_est_at_propr','share_est_at_gov_out',
]]
assert len(COVS_RICH) == 52, f'RICH tem {len(COVS_RICH)}, esperado 52'

SPECS = {'LEAN': COVS_LEAN, 'FULL': COVS_FULL, 'FULL2': COVS_FULL2, 'RICH': COVS_RICH}
for n, c in SPECS.items():
    print(f'  {n:6s}: {len(c)} covs')

# Imputação UF-mediana
all_covs = sorted(set(COVS_RICH))
for c in all_covs:
    if df_cs[c].isna().any():
        df_cs[c] = df_cs.groupby('uf')[c].transform(lambda x: x.fillna(x.median()))
        df_cs[c] = df_cs[c].fillna(df_cs[c].median())
assert df_cs[all_covs].isna().sum().sum() == 0
print(f'\n✓ Imputação OK')

In [ ]:
# Merge ao painel longo
panel_cs = panel.merge(df_cs[['geocode'] + all_covs], on='geocode', how='left')
panel_cs['g_m_cs'] = panel_cs['g_m']  # NaN = never
print(f'panel_cs: {panel_cs.shape}')

OUTCOMES = ['log_produtividade_cana', 'log_emissao_solos_por_ha_agricola']

## Bloco 3 — CS-DR principal: 8 ATTs (4 specs × 2 outcomes)

In [ ]:
import time
cs_results = []
t_total = time.time()

for outcome in OUTCOMES:
    print(f'\n>>> {outcome}')
    for spec_name, covs in SPECS.items():
        t_run = time.time()
        formula = f'{outcome} ~ ' + ' + '.join(covs)
        
        try:
            data = (panel_cs.dropna(subset=[outcome])
                    .set_index(['geocode', 'ano']).sort_index())
            n_munis = data.index.get_level_values('geocode').nunique()
            
            attgt = ATTgt(data=data, cohort_column='g_m_cs')
            attgt.fit(
                formula=formula, est_method='dr', control_group='never_treated',
                boot_iterations=N_BOOT, random_state=RANDOM_STATE,
                progress_bar=False, n_jobs=1,
            )
            
            agg = attgt.aggregate('simple')
            att = float(agg.iloc[0, 0])
            se = float(agg.iloc[0, 1])
            ci_lo = float(agg.iloc[0, 2]) if agg.shape[1] > 2 else att - 1.96 * se
            ci_hi = float(agg.iloc[0, 3]) if agg.shape[1] > 3 else att + 1.96 * se
            
            cs_results.append({
                'outcome': outcome, 'spec': spec_name, 'estimator': 'CS-DR',
                'ATT': att, 'SE': se, 'CI_lo': ci_lo, 'CI_hi': ci_hi,
                'n_munis': n_munis,
            })
            print(f'  {spec_name:6s}  ATT = {att:+.4f} (SE={se:.4f})  [{time.time()-t_run:.1f}s]')
        except Exception as e:
            print(f'  {spec_name:6s}  FALHOU: {type(e).__name__}: {str(e)[:60]}')
            cs_results.append({
                'outcome': outcome, 'spec': spec_name, 'estimator': 'CS-DR',
                'ATT': np.nan, 'SE': np.nan, 'CI_lo': np.nan, 'CI_hi': np.nan,
                'n_munis': 0,
            })

cs_df = pd.DataFrame(cs_results)
print(f'\n✓ CS-DR: {cs_df["ATT"].notna().sum()}/{len(cs_df)} sucessos em {time.time()-t_total:.1f}s')

## Bloco 4 — Sun-Abraham canônico (2021): 2 ATTs (mesma estrutura 11a v4)

In [ ]:
panel_sa = panel.copy()
panel_sa['g_m_int'] = panel_sa['g_m'].fillna(0).astype(int)
panel_sa['event_time'] = np.where(
    panel_sa['g_m_int'] > 0,
    panel_sa['ano'] - panel_sa['g_m_int'],
    -99
)

cohorts_sa = sorted([c for c in panel_sa['g_m_int'].unique() if c > 0])

for g in cohorts_sa:
    for l in range(-7, 6):
        if l == -1:
            continue
        suffix = f'm{abs(l)}' if l < 0 else f'p{l}'
        col = f'D_g{int(g)}_l{suffix}'
        panel_sa[col] = ((panel_sa['g_m_int'] == g) & (panel_sa['event_time'] == l)).astype(int)

dummies_sa = sorted([c for c in panel_sa.columns if c.startswith('D_g')])
dummies_active = [d for d in dummies_sa if panel_sa[d].sum() > 0]

# Pesos cohort (eq. 18)
N_total_sa = (panel_sa[panel_sa['g_m_int'] > 0]
              .groupby('geocode')['g_m_int'].first()).count()
weights_cohort = {}
for g in cohorts_sa:
    N_g = ((panel_sa['g_m_int'] == g).groupby(panel_sa['geocode']).first()).sum()
    weights_cohort[g] = N_g / N_total_sa
print(f'Cohorts: {cohorts_sa}, pesos: {weights_cohort}')

sa_results = []
for outcome in OUTCOMES:
    panel_use = panel_sa.dropna(subset=[outcome]).copy()
    formula = f'{outcome} ~ ' + ' + '.join(dummies_active) + ' | geocode + ano'
    
    try:
        m = pf.feols(formula, data=panel_use, vcov={'CRV1': 'geocode'})
        coefs = m.coef()
        ses = m.se()
        
        att_post_components = []
        for l in range(0, 6):
            suffix = f'p{l}'
            att_l = 0.0
            var_l = 0.0
            for g in cohorts_sa:
                col = f'D_g{int(g)}_l{suffix}'
                if col in coefs.index:
                    att_l += weights_cohort[g] * coefs[col]
                    var_l += (weights_cohort[g] ** 2) * (ses[col] ** 2)
            att_post_components.append({'event_time': l, 'ATT_l': att_l, 'SE_l': np.sqrt(var_l)})
        
        att_arr = np.array([c['ATT_l'] for c in att_post_components])
        se_arr = np.array([c['SE_l'] for c in att_post_components])
        att_agg = att_arr.mean()
        se_agg = np.sqrt((se_arr ** 2).mean())
        
        sa_results.append({
            'outcome': outcome, 'spec': 'sa_canonical', 'estimator': 'Sun-Abraham',
            'ATT': att_agg, 'SE': se_agg,
            'CI_lo': att_agg - 1.96 * se_agg, 'CI_hi': att_agg + 1.96 * se_agg,
            'n_munis': panel_use['geocode'].nunique(),
        })
        print(f'  {outcome:40s} ATT={att_agg:+.4f} (SE={se_agg:.4f})')
    except Exception as e:
        print(f'  {outcome:40s} FALHOU: {str(e)[:60]}')
        sa_results.append({
            'outcome': outcome, 'spec': 'sa_canonical', 'estimator': 'Sun-Abraham',
            'ATT': np.nan, 'SE': np.nan, 'CI_lo': np.nan, 'CI_hi': np.nan, 'n_munis': 0,
        })

sa_df = pd.DataFrame(sa_results)
print(f'\n✓ Sun-Abraham canônico: {sa_df["ATT"].notna().sum()}/{len(sa_df)} sucessos')

## Bloco 5 — TWFE clássico: 2 ATTs

In [ ]:
twfe_results = []

for outcome in OUTCOMES:
    p_t = panel.dropna(subset=[outcome]).copy()
    p_t['g_m_twfe'] = p_t['g_m'].fillna(0).astype(int)
    p_t['post'] = (
        (p_t['ano'] >= p_t['g_m_twfe']).astype(int)
        * (p_t['g_m_twfe'] > 0).astype(int)
    )
    p_t['treated_x_post'] = p_t['is_treated_ever'].astype(int) * p_t['post']
    p_t_idx = p_t.set_index(['geocode', 'ano']).sort_index()
    
    try:
        mod = PanelOLS.from_formula(
            f'{outcome} ~ 1 + treated_x_post + EntityEffects + TimeEffects',
            data=p_t_idx,
        )
        res = mod.fit(cov_type='clustered', cluster_entity=True)
        coef = float(res.params['treated_x_post'])
        se = float(res.std_errors['treated_x_post'])
        
        twfe_results.append({
            'outcome': outcome, 'spec': 'twfe_classic', 'estimator': 'TWFE',
            'ATT': coef, 'SE': se,
            'CI_lo': coef - 1.96 * se, 'CI_hi': coef + 1.96 * se,
            'n_munis': p_t_idx.index.get_level_values('geocode').nunique(),
        })
        print(f'  {outcome:40s} ATT={coef:+.4f} (SE={se:.4f})')
    except Exception as e:
        print(f'  {outcome:40s} FALHOU: {str(e)[:60]}')
        twfe_results.append({
            'outcome': outcome, 'spec': 'twfe_classic', 'estimator': 'TWFE',
            'ATT': np.nan, 'SE': np.nan, 'CI_lo': np.nan, 'CI_hi': np.nan, 'n_munis': 0,
        })

twfe_df = pd.DataFrame(twfe_results)
print(f'\n✓ TWFE: {twfe_df["ATT"].notna().sum()}/{len(twfe_df)} sucessos')

## Bloco 6 — Event-study CS-DR para os 2 outcomes (sob FULL)

In [ ]:
event_results = []
for outcome in OUTCOMES:
    print(f'\n>>> Event-study: {outcome} sob FULL')
    data_es = (panel_cs.dropna(subset=[outcome])
               .set_index(['geocode', 'ano']).sort_index())
    
    attgt_es = ATTgt(data=data_es, cohort_column='g_m_cs')
    attgt_es.fit(
        formula=f'{outcome} ~ ' + ' + '.join(COVS_FULL),
        est_method='dr', control_group='never_treated',
        boot_iterations=N_BOOT, random_state=RANDOM_STATE,
        progress_bar=False, n_jobs=1,
    )
    
    agg_event = attgt_es.aggregate('event')
    
    # Estrutura: index = event_time, columns multi-level (ATT, SE, ...)
    if isinstance(agg_event, pd.DataFrame):
        for idx_val, row in agg_event.iterrows():
            event_results.append({
                'outcome': outcome,
                'event_time': idx_val,
                'ATT': float(row.iloc[0]) if len(row) > 0 else np.nan,
                'SE': float(row.iloc[1]) if len(row) > 1 else np.nan,
            })
    
    print(agg_event)

event_df = pd.DataFrame(event_results)
print(f'\n✓ Event-study: {len(event_df)} pontos estimados')

## Bloco 7 — Salvar e mostrar resumo

In [ ]:
# Consolidar e salvar
all_results = pd.concat([cs_df, sa_df, twfe_df], ignore_index=True)
all_results.to_csv(interim('att_derived_outcomes.csv'), index=False)
event_df.to_csv(interim('att_derived_eventstudy.csv'), index=False)

print(f'✓ att_derived_outcomes.csv: {all_results.shape}')
print(f'✓ att_derived_eventstudy.csv: {event_df.shape}')

# Resumo formatado
print('\n' + '='*80)
print('TABELA — ATTs sobre outcomes derivados (T1 binário)')
print('='*80)

for outcome in OUTCOMES:
    print(f'\n{outcome}:')
    sub = all_results.query('outcome == @outcome').copy()
    for _, row in sub.iterrows():
        if pd.notna(row['ATT']) and pd.notna(row['SE']) and row['SE'] > 0:
            t_stat = abs(row['ATT'] / row['SE'])
            star = '***' if t_stat > 2.58 else ('**' if t_stat > 1.96 else ('*' if t_stat > 1.65 else ''))
            print(f'  {row["estimator"]:12s} {row["spec"]:14s}  ATT={row["ATT"]:+.4f} (SE={row["SE"]:.4f}) {star}')
        else:
            print(f'  {row["estimator"]:12s} {row["spec"]:14s}  FALHOU')

## Interpretação substantiva esperada

**Cenário 1 — Hipótese (c) Expansão de escala não-internalizada GANHA:**
- `log_produtividade_cana` ATT POSITIVO significante (~+5-15%): cana certificada produz mais T/ha
- `log_emissao_solos_por_ha_agricola` ATT NULO ou pequeno: emissão por área agrícola não muda
- **Leitura:** efeito agregado em solos_manejados vem do aumento de produção, não de piora de intensidade. NEEA correta dentro de sua definição (intensidade por MJ), mas falha em capturar efeito de escala.

**Cenário 2 — Hipótese (a) Gap declaração-realidade GANHA:**
- `log_produtividade_cana` ATT NULO ou pequeno: produção/ha não muda
- `log_emissao_solos_por_ha_agricola` ATT POSITIVO significante: emissão por área agrícola sobe
- **Leitura:** piora real de intensidade observada via SEEG, sem aumento de produtividade compensatório. NEEA declarada via RenovaCalc não reflete realidade observada via inventário top-down. Gap declaração-realidade.

**Cenário 3 — Híbrido (parcial de cada):**
- Ambos positivos: tanto escala quanto intensidade pioram. Política está induzindo aumento de produção COM piora de eficiência observada.
- Ambos nulos: efeito em solos_manejados não é robusto ao denominador (improvável dado robustez no 11a/11b).

**Próximo passo recomendado:** se Cenário 1 ganhar, paper EcoEco fica com narrativa de hipótese (c) preferida em §10.3. Se Cenário 2 ganhar, narrativa muda para hipótese (a) preferida — mais crítica à integridade do programa. Cenário 3 mantém as 4 hipóteses como alternativas indistinguíveis.